# 03 — Score the classical methods against the MRI pseudo-labels

Use the MRI-derived labels from `02_mri_pseudo_labels.ipynb` as references for the unsupervised classical methods (`brainseg segment`: otsu, gmm, kmeans) on both brains. Sections come from the slide-deck volumes, because that is the space the labels were warped into (55 µm in-plane, the same pixel size as the 10× low-res sections the methods were tuned on).

**Read every score as agreement with the MRI, not accuracy.** The references are 0.4 mm MRI labels, so boundaries are blocky and the tissue/background edge comes from the MRI brain mask. Fluid is nearly absent from the references (Vlad 1.5% of brain, Moe none), so fluid scores mean little. Gray vs white matter is the informative part. Each method is also scored with a band one MRI voxel wide along every label boundary ignored, which separates boundary blockiness from larger disagreements.

In [ ]:
import csv
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
from PIL import Image

from brainseg.benchmark import run_classical
from brainseg.compare import compare
from brainseg.mri_labels import ignore_boundaries

DATA, OUT = Path("../data"), Path("../outputs")
BENCH = DATA / "benchmark_mri"
SUBJECTS = {"vlad": 0, "moe": 1}  # slide-deck axis that indexes sections
N_SECTIONS = 20
BAND = 4  # pixels: half an MRI voxel (0.2 mm / 0.055 mm), so the ignored band is one voxel wide

## Export sections

Evenly spaced sections between the first and last that are at least half as full of labelled tissue as the fullest section, so nearly empty end sections are skipped. Each row lists retardance then reflectivity. `manifest_band.csv` points at the same sections with the boundary band set to 255.

In [ ]:
rows = []
for s, axis in SUBJECTS.items():
    mapping = {int(k): re.search(r"Slice_\d+", v).group(0) for k, v in json.load(open(DATA / s / "slidedeck_slice_mapping.json")).items()}
    labels = np.asarray(nib.load(OUT / f"{s}_labels_psoct.nii.gz").dataobj)
    decks = {c: nib.load(DATA / s / "warp" / f"{c}_slide_deck.nii.gz") for c in ("Ret", "Ref")}
    tissue = np.isin(labels, (1, 2, 3)).sum(axis=tuple(a for a in range(3) if a != axis))
    full = np.flatnonzero(tissue >= 0.5 * tissue.max())
    picks = np.unique(np.linspace(full[0], full[-1], N_SECTIONS).round().astype(int))
    arrays = {c: np.asarray(d.dataobj, dtype=np.float32) for c, d in decks.items()}
    (BENCH / s).mkdir(parents=True, exist_ok=True)
    for i in picks:
        name = mapping[i]
        files = []
        for c in ("Ret", "Ref"):
            np.save(BENCH / s / f"{name}_{c}.npy", np.take(arrays[c], i, axis))
            files.append(f"{s}/{name}_{c}.npy")
        section = np.take(labels, i, axis).astype(np.uint8)
        np.save(BENCH / s / f"{name}_labels.npy", section)
        np.save(BENCH / s / f"{name}_labels_band.npy", ignore_boundaries(section, BAND))
        rows.append({"sample_id": s, "section_id": name, "split": "test", "image": ";".join(files),
                     "mask": f"{s}/{name}_labels.npy", "pixel_size_mm": 0.055})
    print(f"{s}: {len(picks)} sections, deck indices {picks.min()}-{picks.max()}")
    del arrays

manifests = {"": BENCH / "manifest.csv", "_band": BENCH / "manifest_band.csv"}
for suffix, path in manifests.items():
    with path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows({**r, "mask": r["mask"].replace("_labels", f"_labels{suffix}")} for r in rows)

## Run the methods and compare

In [ ]:
runs = {suffix: [] for suffix in manifests}  # run directories are named <method><suffix>
for suffix, path in manifests.items():
    for method in ("otsu", "gmm", "kmeans"):
        run = OUT / "mri_benchmark" / f"{method}{suffix}"
        run_classical(path, "test", run, method, retardance=0, reflectivity=1, pixel_size=0.055)
        runs[suffix].append(run)

table = compare(runs[""] + runs["_band"], OUT / "mri_benchmark" / "table.csv")
for r in table:
    if r["metric"] == "dice" and r["class"] in ("background", "gray_matter", "white_matter", "fluid_space"):
        ci = f"[{r['ci_low']:.3f}, {r['ci_high']:.3f}]" if r["ci_low"] is not None else ""
        print(f"{r['run']:12s} {r['sample_id']:5s} {r['class']:13s} Dice {r['mean']:.3f} {ci}")

In [ ]:
classes = ("background", "gray_matter", "white_matter")
fig, axes = plt.subplots(2, 2, figsize=(12, 8), layout="constrained", sharey=True)
for (suffix, variant), row in zip((("", "all labels"), ("_band", "boundary band ignored")), axes):
  for ax, s in zip(row, SUBJECTS):
    for k, run in enumerate(runs[suffix]):
        sel = {r["class"]: r for r in table if r["run"] == run.name and r["sample_id"] == s and r["metric"] == "dice"}
        x = np.arange(len(classes)) + (k - 1) * 0.25
        mean = [sel[c]["mean"] for c in classes]
        err = [[m - sel[c]["ci_low"] for m, c in zip(mean, classes)], [sel[c]["ci_high"] - m for m, c in zip(mean, classes)]]
        ax.bar(x, mean, 0.25, yerr=err, label=run.name.removesuffix(suffix) if suffix else run.name, capsize=2)
    ax.set(xticks=range(len(classes)), xticklabels=classes, title=f"{s}, {variant}: Dice (95% CI)", ylim=(0, 1))
axes[0, 0].legend()

Where do the methods and the MRI labels disagree? Pooled confusion over each brain's sections (rows: MRI label; columns: method), as a share of each MRI class.

In [ ]:
names = ["background", "gray", "white", "fluid"]
for run in runs[""]:
    report = json.loads((run / "report.json").read_text())
    for s, m in report["samples"].items():
        cm = np.array(m["confusion_matrix"])[:4, :4]
        share = np.round(100 * cm / np.maximum(cm.sum(1, keepdims=True), 1)).astype(int)
        print(f"{run.name} / {s}\n" + "\n".join(f"  {n:10s} {row}" for n, row in zip(names, share)))

In [ ]:
report = json.loads((runs[""][1] / "report.json").read_text())
fig, axes = plt.subplots(2, 3, figsize=(15, 8), layout="constrained")
for row, s in zip(axes, SUBJECTS):
    secs = [sec for sec in report["sections"] if sec["sample_id"] == s]
    for ax, sec in zip(row, secs[:: max(1, len(secs) // 3)][:3]):
        ret = np.load(BENCH / s / f"{sec['section_id']}_Ret.npy")
        ref = np.load(BENCH / s / f"{sec['section_id']}_labels.npy")
        pred = np.asarray(Image.open(runs[""][1] / sec["prediction"]))
        ax.imshow(np.rot90(ret), cmap="gray", vmin=0, vmax=np.percentile(ret[ret > 0], 99))
        ax.contour(np.rot90(ref == 2), levels=[0.5], colors="tab:orange", linewidths=0.6)
        ax.contour(np.rot90(pred == 2), levels=[0.5], colors="tab:cyan", linewidths=0.6)
        ax.set(xticks=[], yticks=[], title=f"{s} {sec['section_id']}: white matter, MRI (orange) vs gmm (cyan)")

Where are the remaining background disagreements? Distance of MRI-background pixels that gmm calls gray matter from the nearest MRI tissue. Unimaged pixels (zero in both contrasts) are not scored.

In [ ]:
from scipy.ndimage import distance_transform_edt

for s in SUBJECTS:
    near = far = 0
    for sec in (sec for sec in report["sections"] if sec["sample_id"] == s):
        ref = np.load(BENCH / s / f"{sec['section_id']}_labels.npy")
        imaged = (np.load(BENCH / s / f"{sec['section_id']}_Ret.npy") != 0) | (np.load(BENCH / s / f"{sec['section_id']}_Ref.npy") != 0)
        pred = np.asarray(Image.open(runs[""][1] / sec["prediction"]))
        tissue = np.isin(ref, (1, 2, 3))
        disagree = (ref == 0) & (pred == 1) & imaged
        within = distance_transform_edt(~tissue) * 0.055 <= 0.8
        near, far = near + (disagree & within).sum(), far + (disagree & ~within).sum()
    print(f"{s}: background called gray within 0.8 mm of MRI tissue {near / (near + far):.0%}, farther {far / (near + far):.0%}")

## Findings

- **Gray and white matter agree moderately with the MRI labels** on both brains (Dice about 0.67-0.70 gray, 0.75-0.76 white). The headline tissue Dice (~0.49) is pulled down by fluid, which is almost absent from the references, so it should not be quoted on its own.
- **The three methods cannot be told apart** at this level: their intervals overlap. Otsu and k-means are nearly identical; gmm trades gray-matter precision for white-matter recall.
- **The methods trace white matter more finely than the references.** Part of the gap is the references' 0.4 mm blockiness, not method error; this benchmark cannot resolve differences finer than that.
- **The clearest method weakness is the tissue mask.** About a third of MRI background is called gray matter, and most of it lies more than 0.8 mm (two MRI voxels) from any MRI tissue: embedding gel taken for tissue by the Otsu-on-reflectivity mask, not boundary blockiness.
- **Unimaged pixels:** the slide decks mark unimaged pixels with 0, not NaN, so the benchmark scores the few labels that fall there (0.3% of labelled tissue for Vlad, 2.1% for Moe).